# 00 — Data Pipeline

**Purpose:** Fetch all raw data series, construct the gold premium variable, and save one clean CSV.  
**Output:** `../data/gold_policy_clean.csv`  
**Rule:** No charts, no models, no analysis in this notebook. Data only.

---

## Data Dictionary

| Column | Type | Description | Source |
|---|---|---|---|
| `date` | datetime | Trading date (index) | — |
| `Gold_USD` | float | Gold spot price (USD/oz), London PM fix | Yahoo Finance `GC=F` |
| `Silver_USD` | float | Silver spot price (USD/oz) | Yahoo Finance `SI=F` |
| `Oil_USD` | float | Brent crude (USD/bbl) | Yahoo Finance `BZ=F` |
| `rupees_per_dollar` | float | INR/USD spot rate | Yahoo Finance `INR=X` |
| `Gold_INR_actual` | float | IBJA benchmark gold rate (INR/10g, 999 purity, AM fix) | IBJA PDF (May 2022–present); NaN = archive gap or market holiday |
| `Silver_INR_actual` | float | IBJA benchmark silver rate (INR/kg) | IBJA PDF (May 2022–present); NaN = archive gap or market holiday |
| `forex_reserves_usd_bn` | float | RBI total forex reserves (USD billion, weekly, forward-filled) | RBI DBIE |
| `Kalyan` | float | Kalyan Jewellers closing price (INR) | Yahoo Finance `KALYANKJIL.NS` |
| `MCX_Gold` | float | MCX gold futures near-month (INR/10g) — NaN if unavailable | Yahoo Finance / NSE |
| `parity_6pct` | float | Import parity at 6% duty = Gold_USD x (10/31.1035) x FX x 1.06 | Constructed |
| `parity_15pct` | float | Import parity at 15% duty = Gold_USD x (10/31.1035) x FX x 1.15 | Constructed |
| `domestic_premium` | float | IBJA price minus parity_6pct (INR/10g). NaN on IBJA-closed days. | Constructed |
| `premium_pct` | float | domestic_premium / parity_6pct x 100 | Constructed |
| `post_hike` | int | 1 if date >= 2026-05-13, else 0 | Constructed |
| `days_since_hike` | float | Trading days elapsed since May 13 2026. NaN pre-hike. | Constructed |
| `delta_Gold_USD` | float | Day-over-day % change in Gold_USD | Constructed |
| `delta_FX` | float | Day-over-day % change in rupees_per_dollar | Constructed |
| `delta_Oil` | float | Day-over-day % change in Oil_USD | Constructed |
| `ibja_source` | str | `'pdf'` if from IBJA PDF archive; NaN if archive gap or market holiday | Constructed |

---

## Key Dates

| Event | Date |
|---|---|
| Data start | 2022-01-03 |
| Duty cut (15% to 6%) | 2024-07-23 |
| Pre-hike import restriction | 2026-04-02 |
| Duty hike (6% to 15%) | 2026-05-13 |
| Data end | latest available |

---

## Why parity_6pct as the baseline?

We fix the counterfactual duty at 6% (the pre-hike rate) throughout the entire sample.
This means domestic_premium measures: how much extra are buyers paying above what they
would pay if the old duty still applied?

Pre-hike this should hover near zero. Post-hike it should jump by the mechanically implied
amount. Any gap between the actual jump and the theoretical maximum is the pass-through
puzzle we are trying to explain.

In [19]:
# ── Imports ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import pdfplumber
from pathlib import Path
from datetime import date, timedelta
import time
import io

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = DATA_DIR / 'gold_policy_clean.csv'

# ── Key dates ─────────────────────────────────────────────────────────────
START_DATE      = '2022-01-03'   # full history start (needed for ARDL cointegration)
DUTY_CUT_DATE   = '2024-07-23'   # 15% → 6% (Union Budget)
RESTRICTION_DATE = '2026-04-02'  # pre-hike import restriction (new customs/IGST rules)
POLICY_DATE     = '2026-05-13'   # 6% → 15% (treatment date)
END_DATE        = date.today().strftime('%Y-%m-%d')

# ── Parity constants ──────────────────────────────────────────────────────
# Gold is quoted in USD per troy ounce. IBJA quotes INR per 10 grams.
# 1 troy oz = 31.1035 grams  →  10g = 10/31.1035 troy oz
TROY_OZ_TO_10G = 10 / 31.1035

DUTY_OLD = 1.06   # 6%  — pre-hike duty multiplier
DUTY_NEW = 1.15   # 15% — post-hike duty multiplier

# ── Sanity check ─────────────────────────────────────────────────────────
print(f'Date range  : {START_DATE} → {END_DATE}')
print(f'Policy date : {POLICY_DATE}')
print(f'Output      : {OUTPUT_PATH.resolve()}')
print(f'10g per oz  : {TROY_OZ_TO_10G:.6f}')


Date range  : 2022-01-03 → 2026-06-11
Policy date : 2026-05-13
Output      : /Users/olixstudios/Documents/workspace/Projects/gold-policy-project/data/gold_policy_clean.csv
10g per oz  : 0.321507


In [20]:
# ── Cell 2: Fetch Yahoo Finance series ───────────────────────────────────
import logging
logging.getLogger('yfinance').setLevel(logging.CRITICAL)  # suppress yfinance noise

YF_TICKERS = {
    'GC=F':           'Gold_USD',
    'SI=F':           'Silver_USD',
    'BZ=F':           'Oil_USD',
    'INR=X':          'rupees_per_dollar',
    'KALYANKJIL.NS':  'Kalyan',
    '^NSEI':          'Nifty50',
    '^TNX':           'US10Y_yield',
}

# Tickers attempted separately (may not be available on Yahoo)
MCX_TICKERS_TO_TRY = ['GOLD.MCX', 'GOLDM.MCX']   # try in order, use first that works
GOLDBEES_TICKER    = 'GOLDBEES.NS'                # India gold ETF

# ── Bulk fetch ────────────────────────────────────────────────────────────
print('Fetching Yahoo Finance data...')
raw = yf.download(
    list(YF_TICKERS.keys()),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

df_yf = raw['Close'].copy()
df_yf.columns    = [YF_TICKERS[c] for c in df_yf.columns]
df_yf.index      = pd.to_datetime(df_yf.index)
df_yf.index.name = 'date'

# ── MCX Gold (try alternatives) ───────────────────────────────────────────
df_yf['MCX_Gold'] = np.nan
for ticker in MCX_TICKERS_TO_TRY:
    try:
        mcx_raw = yf.download(ticker, start=START_DATE, end=END_DATE,
                              auto_adjust=True, progress=False)
        if not mcx_raw.empty and mcx_raw['Close'].notna().sum() > 10:
            df_yf['MCX_Gold'] = mcx_raw['Close'].reindex(df_yf.index)
            print(f'  MCX Gold  : {ticker} — {df_yf["MCX_Gold"].notna().sum()} rows')
            break
    except Exception:
        continue
else:
    print('  MCX Gold  : not available on Yahoo Finance — column will be NaN')

# ── GOLDBEES (India gold ETF) ─────────────────────────────────────────────
df_yf['GOLDBEES'] = np.nan
try:
    gb_raw = yf.download(GOLDBEES_TICKER, start=START_DATE, end=END_DATE,
                         auto_adjust=True, progress=False)
    if not gb_raw.empty:
        df_yf['GOLDBEES'] = gb_raw['Close'].reindex(df_yf.index)
        print(f'  GOLDBEES  : {df_yf["GOLDBEES"].notna().sum()} rows')
    else:
        print('  GOLDBEES  : empty response — column set to NaN')
except Exception as e:
    print(f'  GOLDBEES  : fetch failed — column set to NaN')

# ── Report ────────────────────────────────────────────────────────────────
print(f'\nYahoo Finance fetch complete')
print(f'  Shape      : {df_yf.shape}')
print(f'  Date range : {df_yf.index.min().date()} to {df_yf.index.max().date()}')
print(f'\nNaN counts (expected: low for Gold/FX, moderate for equity series):')
nan_summary = df_yf.isna().sum().rename('NaNs').to_frame()
nan_summary['pct'] = (nan_summary['NaNs'] / len(df_yf) * 100).round(1).astype(str) + '%'
print(nan_summary.to_string())


Fetching Yahoo Finance data...
  MCX Gold  : not available on Yahoo Finance — column will be NaN
  GOLDBEES  : 1095 rows

Yahoo Finance fetch complete
  Shape      : (1157, 9)
  Date range : 2022-01-03 to 2026-06-10

NaN counts (expected: low for Gold/FX, moderate for equity series):
                   NaNs     pct
Oil_USD              41    3.5%
Gold_USD             42    3.6%
rupees_per_dollar     4    0.3%
Kalyan               61    5.3%
Silver_USD           42    3.6%
Nifty50              64    5.5%
US10Y_yield          44    3.8%
MCX_Gold           1157  100.0%
GOLDBEES             62    5.4%


In [21]:
# ── Cell 3: IBJA PDF Scraper ─────────────────────────────────────────────
# Fetches IBJA Daily Bullion Report PDFs and extracts Gold 999 & Silver 999 AM prices.
#
# Date note: IBJA publishes the PDF on day D with rates FROM day D-1.
# We tag each row with the FILENAME date. A 1-day offset correction
# is applied during alignment (Cell 6) — not here.

import pdfplumber, requests, io, time
from datetime import datetime, timedelta
import pandas as pd

IBJA_BASE = 'https://ibja.co/Upload/IBJA_Bullion%20Daily%20Report%20-%20{}.pdf'
HEADERS   = {'User-Agent': 'Mozilla/5.0'}

# ── Helper functions ──────────────────────────────────────────────────────

def make_ibja_url(dt):
    return IBJA_BASE.format(dt.strftime('%d-%m-%Y'))

def parse_ibja_tables(tables):
    """
    Finds Table 1 — the main rates table with headers:
    ['Description', 'Purity', 'AM', 'PM']
    Returns (gold_999_am, silver_999_am) as floats or (None, None).
    """
    for table in tables:
        if not table or len(table) < 2:
            continue
        headers = [str(h).strip().upper() for h in table[0]]
        if 'DESCRIPTION' in headers and 'PURITY' in headers and 'AM' in headers:
            gold_am, silver_am = None, None
            for row in table[1:]:
                if len(row) < 3:
                    continue
                desc   = str(row[0]).strip().upper()
                purity = str(row[1]).strip()
                am_raw = str(row[2]).strip().replace(',', '')
                try:
                    val = float(am_raw)
                except ValueError:
                    continue
                if desc == 'GOLD' and purity == '999':
                    gold_am = val
                elif desc == 'SILVER' and purity == '999':
                    silver_am = val
            if gold_am is not None:
                return gold_am, silver_am
    return None, None

def fetch_ibja_day(dt):
    """
    Fetches and parses one date. Returns (gold_am, silver_am) or (None, None).
    """
    try:
        r = requests.get(make_ibja_url(dt), headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None, None
        with pdfplumber.open(io.BytesIO(r.content)) as pdf:
            tables = pdf.pages[0].extract_tables()
        return parse_ibja_tables(tables)
    except Exception:
        return None, None

# ── Step 1: Verify parser on known date ───────────────────────────────────
TEST_DATE = datetime(2026, 1, 30)
g, s = fetch_ibja_day(TEST_DATE)
print(f'Parser test [{TEST_DATE.date()}]:')
print(f'  Gold 999 AM  : {g} INR/10g')
print(f'  Silver 999 AM: {s} INR/kg')
assert g == 176121.0, f'Unexpected gold value: {g}'
print('  Parser OK.')

# ── Step 2: Loop over all dates ───────────────────────────────────────────
# Loops Mon-Fri from START_DATE to today.
# Skips weekends (IBJA never publishes on weekends).
# Sleeps 0.3s between requests — polite scraping.
# Expected runtime: ~8-12 min for full 2022-2026 range.

start = datetime.strptime(START_DATE, '%Y-%m-%d')
end   = datetime.today()

records = []
dt = start
total_days = 0
found = 0

print(f'\nStarting IBJA fetch: {start.date()} to {end.date()}')
print('Progress printed every 100 dates...\n')

while dt <= end:
    if dt.weekday() < 5:   # Mon=0 … Fri=4, skip Sat/Sun
        total_days += 1
        gold_am, silver_am = fetch_ibja_day(dt)
        records.append({
            'date'            : dt.date(),
            'Gold_INR_actual' : gold_am,
            'Silver_INR_actual': silver_am,
            'ibja_source'     : 'pdf' if gold_am else None
        })
        if gold_am:
            found += 1
        if total_days % 100 == 0:
            print(f'  {dt.date()} — {total_days} weekdays processed, {found} rates found')
        time.sleep(0.3)
    dt += timedelta(days=1)

df_ibja = pd.DataFrame(records).set_index('date')
df_ibja.index = pd.to_datetime(df_ibja.index)

n_total  = len(df_ibja)
n_found  = df_ibja['Gold_INR_actual'].notna().sum()
n_missing = n_total - n_found

print(f'\nIBJA PDF scrape complete:')
print(f'  Weekdays checked : {n_total}')
print(f'  Rates found (PDF): {n_found}')
print(f'  Missing (NaN)    : {n_missing}  ← these go to GoodReturns fallback')
print(f'\nSample:')
print(df_ibja[df_ibja["Gold_INR_actual"].notna()].head(3))


Parser test [2026-01-30]:
  Gold 999 AM  : 176121.0 INR/10g
  Silver 999 AM: 385933.0 INR/kg
  Parser OK.

Starting IBJA fetch: 2022-01-03 to 2026-06-11
Progress printed every 100 dates...

  2022-05-20 — 100 weekdays processed, 9 rates found
  2022-10-07 — 200 weekdays processed, 103 rates found
  2023-02-24 — 300 weekdays processed, 195 rates found
  2023-07-14 — 400 weekdays processed, 266 rates found
  2023-12-01 — 500 weekdays processed, 306 rates found
  2024-04-19 — 600 weekdays processed, 396 rates found
  2024-09-06 — 700 weekdays processed, 455 rates found
  2025-01-24 — 800 weekdays processed, 509 rates found
  2025-06-13 — 900 weekdays processed, 601 rates found
  2025-10-31 — 1000 weekdays processed, 668 rates found
  2026-03-20 — 1100 weekdays processed, 756 rates found

IBJA PDF scrape complete:
  Weekdays checked : 1159
  Rates found (PDF): 809
  Missing (NaN)    : 350  ← these go to GoodReturns fallback

Sample:
            Gold_INR_actual  Silver_INR_actual ibja_sourc

In [24]:
# ── Cell 3b TEST: Probe GoodReturns URL structure ─────────────────────────
# Run this cell FIRST. We inspect the raw HTML before building the loop.
# Goal: confirm URL format and table structure for a known-NaN date.

import requests
from bs4 import BeautifulSoup

HEADERS_GR = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml',
    'Accept-Language': 'en-IN,en;q=0.9',
}

# Jan 3, 2022 — our START_DATE, definitely NaN from Cell 3
test_dt = pd.Timestamp('2022-01-03')

# GoodReturns URL patterns to try (try each until one gives status 200)
url_candidates = [
    f'https://www.goodreturns.in/gold-rates/{test_dt.year}/{test_dt.strftime("%B").lower()}/',
    f'https://www.goodreturns.in/gold-rates/',
    f'https://www.goodreturns.in/gold-rates/gold-price-in-india-today.html',
]

for url in url_candidates:
    r = requests.get(url, headers=HEADERS_GR, timeout=15)
    print(f'URL   : {url}')
    print(f'Status: {r.status_code}  |  Bytes: {len(r.content)}')
    if r.status_code == 200:
        soup = BeautifulSoup(r.content, 'html.parser')
        tables = soup.find_all('table')
        print(f'Tables found: {len(tables)}')
        # Print first 3 table headers so we can identify the rates table
        for i, t in enumerate(tables[:4]):
            rows = t.find_all('tr')
            if rows:
                print(f'  Table {i} — first row: {rows[0].get_text(separator=" | ", strip=True)[:120]}')
                if len(rows) > 1:
                    print(f'  Table {i} — 2nd row : {rows[1].get_text(separator=" | ", strip=True)[:120]}')
        break
    print()

URL   : https://www.goodreturns.in/gold-rates/2022/january/
Status: 404  |  Bytes: 489262

URL   : https://www.goodreturns.in/gold-rates/
Status: 200  |  Bytes: 613383
Tables found: 4
  Table 0 — first row: Gram | 24K | 22K | 18K
  Table 0 — 2nd row : 1 | ₹14,886 | (-430) | ₹13,645 | (-395) | ₹11,164 | (-323)
  Table 1 — first row: City | 24K | 22K | 18K
  Table 1 — 2nd row : Chennai | ₹15,055 | ₹13,800 | ₹11,570
  Table 2 — first row: Country | CURRENCY | 24K | 22K | 18K | 24K (INR)
  Table 2 — 2nd row : Bahrain | BHD | 52.40 | 48.30 | 39.50 | ₹13,268
  Table 3 — first row: Date | 24K | 22K
  Table 3 — 2nd row : Jun 10, 2026 | ₹14,886 | (-430) | ₹13,645 | (-395)


In [25]:
# ── Cell 3b DIAGNOSTIC: How far back does Table 3 go? ────────────────────
import requests
from bs4 import BeautifulSoup
import pandas as pd

HEADERS_GR = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml',
    'Accept-Language': 'en-IN,en;q=0.9',
}

r = requests.get('https://www.goodreturns.in/gold-rates/', headers=HEADERS_GR, timeout=15)
soup = BeautifulSoup(r.content, 'html.parser')
tables = soup.find_all('table')

# Table 3 is the historical rates table
t3 = tables[3]
rows = t3.find_all('tr')
print(f'Table 3 — total rows (including header): {len(rows)}')
print(f'\nFirst 3 data rows:')
for row in rows[1:4]:
    print(' | '.join(c.get_text(strip=True) for c in row.find_all(['td','th'])))
print(f'\nLast 3 data rows:')
for row in rows[-3:]:
    print(' | '.join(c.get_text(strip=True) for c in row.find_all(['td','th'])))

# Also check if there's a "load more" or pagination link
pagination = soup.find_all(attrs={'class': lambda c: c and 'page' in c.lower()})
more_links = [a['href'] for a in soup.find_all('a', href=True) 
              if any(kw in a['href'].lower() for kw in ['page=', 'history', 'older', 'archive'])]
print(f'\nPagination/older links found: {more_links[:5]}')

# Also test the URL 1 content — it returned 489KB despite 404 status
# (some sites serve full pages with wrong status codes)
r2 = requests.get('https://www.goodreturns.in/gold-rates/2022/january/', 
                   headers=HEADERS_GR, timeout=15)
soup2 = BeautifulSoup(r2.content, 'html.parser')
tables2 = soup2.find_all('table')
print(f'\nURL-1 "404" page — tables found: {len(tables2)}')
if tables2:
    rows2 = tables2[0].find_all('tr')
    print(f'Table 0, row count: {len(rows2)}')
    if rows2:
        print(f'Table 0, header: {rows2[0].get_text(separator=" | ", strip=True)[:100]}')

Table 3 — total rows (including header): 11

First 3 data rows:
Jun 10, 2026 | ₹14,886(-430) | ₹13,645(-395)
Jun 09, 2026 | ₹15,316(+147) | ₹14,040(+135)
Jun 08, 2026 | ₹15,169(-104) | ₹13,905(-95)

Last 3 data rows:
Jun 03, 2026 | ₹15,622(0) | ₹14,320(0)
Jun 02, 2026 | ₹15,622(0) | ₹14,320(0)
Jun 01, 2026 | ₹15,622(-82) | ₹14,320(-75)

Pagination/older links found: []

URL-1 "404" page — tables found: 0


In [26]:
# ── Cell 3b PROBE v2: Multi-source historical IBJA data ──────────────────
import requests
from bs4 import BeautifulSoup

HEADERS_GR = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml',
    'Accept-Language': 'en-IN,en;q=0.9',
}

candidates = [
    ('GoodReturns Mumbai',     'https://www.goodreturns.in/gold-rates/gold-rates-in-mumbai.html'),
    ('GoodReturns Historical', 'https://www.goodreturns.in/gold-rates-history/2022/'),
    ('goldpriceindia IBJA',    'https://goldpriceindia.com/ibja-gold-rate-today/'),
    ('goldpriceindia History', 'https://goldpriceindia.com/gold-rate-today-history/'),
    ('BankBazaar History',     'https://www.bankbazaar.com/gold-rate/historical-gold-price-india.html'),
]

for name, url in candidates:
    try:
        r = requests.get(url, headers=HEADERS_GR, timeout=15)
        soup = BeautifulSoup(r.content, 'html.parser')
        tables = soup.find_all('table')
        
        # Find the table with the most rows (likely the data table)
        if tables:
            biggest = max(tables, key=lambda t: len(t.find_all('tr')))
            rows = biggest.find_all('tr')
            n_rows = len(rows)
            first_data = rows[1].get_text(separator=' | ', strip=True)[:80] if len(rows) > 1 else 'n/a'
            last_data  = rows[-1].get_text(separator=' | ', strip=True)[:80] if n_rows > 2 else 'n/a'
        else:
            n_rows, first_data, last_data = 0, 'no tables', 'no tables'
        
        print(f'[{name}]')
        print(f'  Status : {r.status_code}  |  Bytes: {len(r.content):,}')
        print(f'  Tables : {len(tables)}  |  Biggest table: {n_rows} rows')
        print(f'  First  : {first_data}')
        print(f'  Last   : {last_data}')
        print()
    except Exception as e:
        print(f'[{name}] ERROR: {e}\n')

[GoodReturns Mumbai]
  Status : 404  |  Bytes: 532,482
  Tables : 0  |  Biggest table: 0 rows
  First  : no tables
  Last   : no tables

[GoodReturns Historical]
  Status : 404  |  Bytes: 489,262
  Tables : 0  |  Biggest table: 0 rows
  First  : no tables
  Last   : no tables

[goldpriceindia IBJA]
  Status : 404  |  Bytes: 2,061
  Tables : 0  |  Biggest table: 0 rows
  First  : no tables
  Last   : no tables

[goldpriceindia History]
  Status : 404  |  Bytes: 2,064
  Tables : 0  |  Biggest table: 0 rows
  First  : no tables
  Last   : no tables

[BankBazaar History]
  Status : 404  |  Bytes: 79,836
  Tables : 0  |  Biggest table: 0 rows
  First  : no tables
  Last   : no tables



In [27]:
# ── Cell 3b PROBE v3 ──────────────────────────────────────────────────────
candidates_v3 = [
    ('GoodReturns Mumbai-2',  'https://www.goodreturns.in/gold-rates/mumbai/'),
    ('GoodReturns Mumbai-3',  'https://www.goodreturns.in/gold-rates/gold-rate-in-mumbai/'),
    ('MoneyControl Gold',     'https://www.moneycontrol.com/commodity/gold-price.html'),
    ('GoldRate24 India',      'https://www.goldrate24.com/gold-rates/india/'),
    ('IBJA Rate Page',        'https://ibja.co/Rates/'),
]

for name, url in candidates_v3:
    try:
        r = requests.get(url, headers=HEADERS_GR, timeout=15)
        soup = BeautifulSoup(r.content, 'html.parser')
        tables = soup.find_all('table')
        
        if tables:
            biggest = max(tables, key=lambda t: len(t.find_all('tr')))
            rows = biggest.find_all('tr')
            n_rows = len(rows)
            first_data = rows[1].get_text(separator=' | ', strip=True)[:80] if len(rows) > 1 else 'n/a'
            last_data  = rows[-1].get_text(separator=' | ', strip=True)[:80] if n_rows > 2 else 'n/a'
        else:
            n_rows, first_data, last_data = 0, 'no tables', 'no tables'
        
        print(f'[{name}]')
        print(f'  Status : {r.status_code}  |  Bytes: {len(r.content):,}')
        print(f'  Tables : {len(tables)}  |  Biggest: {n_rows} rows')
        print(f'  First  : {first_data}')
        print(f'  Last   : {last_data}')
        print()
    except Exception as e:
        print(f'[{name}] ERROR: {e}\n')

[GoodReturns Mumbai-2]
  Status : 404  |  Bytes: 489,262
  Tables : 0  |  Biggest: 0 rows
  First  : no tables
  Last   : no tables

[GoodReturns Mumbai-3]
  Status : 404  |  Bytes: 489,262
  Tables : 0  |  Biggest: 0 rows
  First  : no tables
  Last   : no tables

[MoneyControl Gold]
  Status : 403  |  Bytes: 409
  Tables : 0  |  Biggest: 0 rows
  First  : no tables
  Last   : no tables

[GoldRate24 India]
  Status : 200  |  Bytes: 118,099
  Tables : 12  |  Biggest: 55 rows
  First  : Gold Rate in Ethiopia (ETB)
  Last   : All Countries

[IBJA Rate Page]
  Status : 404  |  Bytes: 4,839
  Tables : 2  |  Biggest: 4 rows
  First  : Notification | MapRequestHandler
  Last   : Error Code | 0x80070002



In [28]:
# ── Cell 3b PROBE v4: Inspect all GoldRate24 tables ──────────────────────
r = requests.get('https://www.goldrate24.com/gold-rates/india/', headers=HEADERS_GR, timeout=15)
soup = BeautifulSoup(r.content, 'html.parser')
tables = soup.find_all('table')

print(f'Total tables: {len(tables)}\n')
for i, t in enumerate(tables):
    rows = t.find_all('tr')
    n = len(rows)
    header = rows[0].get_text(separator=' | ', strip=True)[:100] if rows else 'empty'
    first  = rows[1].get_text(separator=' | ', strip=True)[:100] if n > 1 else 'n/a'
    last   = rows[-1].get_text(separator=' | ', strip=True)[:100] if n > 2 else 'n/a'
    print(f'Table {i:02d} — {n} rows')
    print(f'  Header : {header}')
    print(f'  First  : {first}')
    print(f'  Last   : {last}')
    print()

# Also try GoldRate24 India historical URL variants
print('--- URL variants ---')
variants = [
    'https://www.goldrate24.com/gold-rates/india/history/',
    'https://www.goldrate24.com/gold-rates/india/historical-gold-rates/',
    'https://www.goldrate24.com/gold-rates/india/?year=2022',
    'https://www.goldrate24.com/gold-price/india/history/',
]
for url in variants:
    try:
        r2 = requests.get(url, headers=HEADERS_GR, timeout=10)
        soup2 = BeautifulSoup(r2.content, 'html.parser')
        tables2 = soup2.find_all('table')
        biggest2 = max(tables2, key=lambda t: len(t.find_all('tr'))) if tables2 else None
        n2 = len(biggest2.find_all('tr')) if biggest2 else 0
        first2 = biggest2.find_all('tr')[1].get_text(separator=' | ', strip=True)[:80] if n2 > 1 else 'n/a'
        last2  = biggest2.find_all('tr')[-1].get_text(separator=' | ', strip=True)[:80] if n2 > 2 else 'n/a'
        print(f'{r2.status_code} | {url}')
        print(f'  Tables: {len(tables2)} | Biggest: {n2} rows | First: {first2}')
    except Exception as e:
        print(f'ERR | {url}: {e}')

Total tables: 12

Table 00 — 13 rows
  Header : Gold Unit | Gold Price in Indian Rupee (INR) | Gold Price in U.S. Dollar (USD)
  First  : Gold Ounce | 438,878.24 | 4,070.94
  Last   : Gold Gram 8K | 4,699.24 | 43.59

Table 01 — 11 rows
  Header : Gold Price in Indian Rupee (INR) | Gold Price in U.S. Dollar (USD)
  First  : Gold Ounce 24K | 438,878.24 | 4,070.94
  Last   : Gold Ounce 8K | 146,146.46 | 1,355.62

Table 02 — 11 rows
  Header : Gold Price in Indian Rupee (INR) | Gold Price in U.S. Dollar (USD)
  First  : Gold Gram 24K | 14,109.94 | 130.88
  Last   : Gold Gram 8K | 4,698.61 | 43.58

Table 03 — 11 rows
  Header : Gold Price in Indian Rupee (INR) | Gold Price in U.S. Dollar (USD)
  First  : Gold Kilogram 24K | 14,109,935.57 | 130,880.72
  Last   : Gold Kilogram 8K | 4,698,608.54 | 43,583.28

Table 04 — 11 rows
  Header : Gold Price in Indian Rupee (INR) | Gold Price in U.S. Dollar (USD)
  First  : Gold Tola 24K | 164,579.34 | 1,526.60
  Last   : Gold Tola 8K | 54,804.92 | 508.

In [30]:
# ── Cell 4 TEST: Probe RBI forex reserves sources ────────────────────────
import requests
from bs4 import BeautifulSoup

HEADERS_RBI = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/json,*/*',
    'Accept-Language': 'en-IN,en;q=0.9',
}

rbi_candidates = [
    ('RBI Statistics landing',   'https://www.rbi.org.in/Scripts/Statistics.aspx'),
    ('RBI DBIE portal',          'https://dbie.rbi.org.in/DBIE/dbie.rbi'),
    ('RBI Data Warehouse',       'https://data.rbi.org.in/'),
    ('RBI Press Release index',  'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx'),
    ('RBI recent FX PR',         'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=58700'),
]

for name, url in rbi_candidates:
    try:
        r = requests.get(url, headers=HEADERS_RBI, timeout=15)
        soup = BeautifulSoup(r.content, 'html.parser')
        tables = soup.find_all('table')
        
        if tables:
            biggest = max(tables, key=lambda t: len(t.find_all('tr')))
            rows = biggest.find_all('tr')
            n_rows = len(rows)
            header = rows[0].get_text(separator=' | ', strip=True)[:100] if rows else 'n/a'
            first  = rows[1].get_text(separator=' | ', strip=True)[:100] if n_rows > 1 else 'n/a'
            last   = rows[-1].get_text(separator=' | ', strip=True)[:100] if n_rows > 2 else 'n/a'
        else:
            n_rows, header, first, last = 0, 'no tables', '', ''

        # Also look for download links (.csv, .xlsx, .xls)
        dl_links = [a['href'] for a in soup.find_all('a', href=True)
                    if any(ext in a['href'].lower() for ext in ['.csv', '.xlsx', '.xls', 'download'])]

        print(f'[{name}]')
        print(f'  Status : {r.status_code}  |  Bytes: {len(r.content):,}')
        print(f'  Tables : {len(tables)}  |  Biggest: {n_rows} rows')
        if n_rows > 0:
            print(f'  Header : {header}')
            print(f'  First  : {first}')
            print(f'  Last   : {last}')
        if dl_links:
            print(f'  Downloads: {dl_links[:3]}')
        print()
    except Exception as e:
        print(f'[{name}] ERROR: {e}\n')

[RBI Statistics landing]
  Status : 200  |  Bytes: 97,638
  Tables : 1  |  Biggest: 86 rows
  Header : Daily
  First  : LAF Result
  Last   : Survey on Foreign Collaboration in Indian Industry (FCS)
  Downloads: ['https://rbidocs.rbi.org.in/rdocs/content/docs/PSDDP04062020.xlsx']

[RBI DBIE portal] ERROR: HTTPSConnectionPool(host='dbie.rbi.org.in', port=443): Max retries exceeded with url: /DBIE/dbie.rbi (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'dbie.rbi.org.in'. (_ssl.c:1006)")))

[RBI Data Warehouse]
  Status : 200  |  Bytes: 50,732
  Tables : 0  |  Biggest: 0 rows

[RBI Press Release index]
  Status : 200  |  Bytes: 172,884
  Tables : 2  |  Biggest: 72 rows
  Header : Jun 10, 2026
  First  : Conversion/Switch of Government of India Securities | 220 kb
  Last   : Money Market Operations as on June 01, 2026 | 494 kb

[RBI recent FX PR]
  Status : 200  |  Bytes: 155,802
 

In [31]:
# ── Combined probe: RBI portals + data.gov.in ────────────────────────────
import requests
from bs4 import BeautifulSoup

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/json,*/*',
    'Accept-Language': 'en-IN,en;q=0.9',
}

candidates = [
    # RBI portals
    ('RBI Statistics landing',  'https://www.rbi.org.in/Scripts/Statistics.aspx'),
    ('RBI DBIE portal',         'https://dbie.rbi.org.in/DBIE/dbie.rbi'),
    ('RBI Data Warehouse',      'https://data.rbi.org.in/'),
    ('RBI Press Release index', 'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx'),
    ('RBI recent FX PR',        'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=58700'),
    # data.gov.in
    ('OGD gold search',         'https://www.data.gov.in/search?title=gold&type=resources&sortby=_score'),
    ('OGD forex search',        'https://www.data.gov.in/search?title=foreign+exchange+reserves&type=resources&sortby=_score'),
    ('OGD IBJA search',         'https://www.data.gov.in/search?title=IBJA&type=resources&sortby=_score'),
    ('OGD gold import search',  'https://www.data.gov.in/search?title=gold+import&type=resources&sortby=_score'),
    # data.gov.in CKAN-style API
    ('OGD API gold',            'https://data.gov.in/api/3/action/resource_search?query=title:gold'),
]

for name, url in candidates:
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(r.content, 'html.parser')
        tables = soup.find_all('table')

        if tables:
            biggest = max(tables, key=lambda t: len(t.find_all('tr')))
            rows = biggest.find_all('tr')
            n_rows = len(rows)
            header = rows[0].get_text(separator=' | ', strip=True)[:100] if rows else 'n/a'
            first  = rows[1].get_text(separator=' | ', strip=True)[:100] if n_rows > 1 else 'n/a'
        else:
            n_rows, header, first = 0, '', ''

        # Download links (.csv, .xlsx, .json) — key for data.gov.in
        dl_links = [a['href'] for a in soup.find_all('a', href=True)
                    if any(ext in a['href'].lower() for ext in ['.csv', '.xlsx', '.xls', '.json', 'download', 'resource'])]

        # For JSON responses (API endpoints)
        is_json = 'json' in r.headers.get('Content-Type', '')
        json_preview = ''
        if is_json:
            try:
                j = r.json()
                json_preview = str(j)[:200]
            except Exception:
                pass

        print(f'[{name}]')
        print(f'  Status : {r.status_code}  |  Bytes: {len(r.content):,}')
        print(f'  Tables : {len(tables)}  |  Biggest: {n_rows} rows')
        if header:
            print(f'  Header : {header}')
        if first:
            print(f'  First  : {first}')
        if dl_links:
            print(f'  Downloads: {dl_links[:4]}')
        if json_preview:
            print(f'  JSON   : {json_preview}')
        print()
    except Exception as e:
        print(f'[{name}] ERROR: {e}\n')

[RBI Statistics landing]
  Status : 200  |  Bytes: 97,638
  Tables : 1  |  Biggest: 86 rows
  Header : Daily
  First  : LAF Result
  Downloads: ['https://rbidocs.rbi.org.in/rdocs/content/docs/PSDDP04062020.xlsx']

[RBI DBIE portal] ERROR: HTTPSConnectionPool(host='dbie.rbi.org.in', port=443): Max retries exceeded with url: /DBIE/dbie.rbi (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'dbie.rbi.org.in'. (_ssl.c:1006)")))

[RBI Data Warehouse]
  Status : 200  |  Bytes: 50,732
  Tables : 0  |  Biggest: 0 rows

[RBI Press Release index]
  Status : 200  |  Bytes: 172,884
  Tables : 2  |  Biggest: 72 rows
  Header : Jun 10, 2026
  First  : Conversion/Switch of Government of India Securities | 220 kb

[RBI recent FX PR]
  Status : 200  |  Bytes: 155,802
  Tables : 8  |  Biggest: 87 rows
  Header : ( | 425 kb | )
  First  : Date : Sep 13, 2024

[OGD gold search]
  Status : 200  |  Byte

In [32]:
# ── Cell 4 PROBE v2: Parse RBI press release index ───────────────────────
import requests
from bs4 import BeautifulSoup
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
}

# ── Step 1: Parse the press release index for forex reserve entries ───────
r = requests.get(
    'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx',
    headers=HEADERS, timeout=15
)
soup = BeautifulSoup(r.content, 'html.parser')
tables = soup.find_all('table')

# Find the index table (72 rows)
index_table = max(tables, key=lambda t: len(t.find_all('tr')))
rows = index_table.find_all('tr')

fx_entries = []
for row in rows:
    text = row.get_text(' ', strip=True)
    link = row.find('a', href=True)
    if link and 'foreign exchange reserves' in text.lower():
        href = link['href']
        prid_match = re.search(r'prid=(\d+)', href, re.IGNORECASE)
        prid = prid_match.group(1) if prid_match else 'unknown'
        fx_entries.append({'text': text[:80], 'href': href, 'prid': prid})

print(f'Forex reserve entries in index: {len(fx_entries)}')
for e in fx_entries[:5]:
    print(f"  prid={e['prid']} | {e['text']}")

# ── Step 2: Fetch one forex reserve press release and inspect tables ──────
if fx_entries:
    target_prid = fx_entries[0]['prid']
else:
    target_prid = '58700'   # fallback to the one we know works

pr_url = f'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid={target_prid}'
print(f'\nFetching: {pr_url}')
r2 = requests.get(pr_url, headers=HEADERS, timeout=15)
soup2 = BeautifulSoup(r2.content, 'html.parser')
tables2 = soup2.find_all('table')
print(f'Tables found: {len(tables2)}')

for i, t in enumerate(tables2):
    rows2 = t.find_all('tr')
    n = len(rows2)
    header = rows2[0].get_text(separator=' | ', strip=True)[:100] if rows2 else 'empty'
    first  = rows2[1].get_text(separator=' | ', strip=True)[:100] if n > 1 else 'n/a'
    last   = rows2[-1].get_text(separator=' | ', strip=True)[:100] if n > 2 else 'n/a'
    print(f'\nTable {i:02d} — {n} rows')
    print(f'  Header : {header}')
    print(f'  First  : {first}')
    print(f'  Last   : {last}')

Forex reserve entries in index: 1
  prid=62891 | Sources of Variation in India’s Foreign Exchange Reserves during April-March 202

Fetching: https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=62891
Tables found: 5

Table 00 — 29 rows
  Header : ( | 325 kb | )
  First  : Date : Jun 08, 2026
  Last   : Note: Increase in reserves (+)/Decrease in reserves (-). | Difference, if any, is due to rounding of

Table 01 — 29 rows
  Header : ( | 325 kb | )
  First  : Date : Jun 08, 2026
  Last   : Note: Increase in reserves (+)/Decrease in reserves (-). | Difference, if any, is due to rounding of

Table 02 — 25 rows
  Header : Today, the Reserve Bank of India released the | balance of payments (BoP) data for the fourth quarte
  First  : Table 1: Sources of Variation in Foreign Exchange Reserves*
  Last   : Note: Increase in reserves (+)/Decrease in reserves (-). | Difference, if any, is due to rounding of

Table 03 — 17 rows
  Header : Table 1: Sources of Variation in Foreign Exchange

In [33]:
# ── Cell 4 PROBE v3: Scan prid range for weekly forex reserve releases ────
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {'User-Agent': 'Mozilla/5.0'}

found_fx = []

# Scan downward from 62891 — weekly releases should be nearby
for prid in range(62890, 62840, -1):
    url = f'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid={prid}'
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            continue
        soup = BeautifulSoup(r.content, 'html.parser')
        
        # RBI press release title is usually in an <h2> or the first prominent heading
        title_tag = soup.find('h2') or soup.find('h3') or soup.find('title')
        title = title_tag.get_text(strip=True)[:120] if title_tag else 'no title'
        
        if any(kw in title.lower() for kw in ['foreign exchange reserve', 'forex reserve', 'week ended']):
            found_fx.append({'prid': prid, 'title': title})
            print(f'✅ prid={prid}: {title}')
        else:
            print(f'   prid={prid}: {title[:80]}')
        
        time.sleep(0.2)
    except Exception as e:
        print(f'   prid={prid}: ERROR {e}')

print(f'\nFound {len(found_fx)} forex reserve weekly releases in range 62840-62890')

# If we found one, inspect its table structure
if found_fx:
    target = found_fx[0]['prid']
    r2 = requests.get(
        f'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid={target}',
        headers=HEADERS, timeout=15
    )
    soup2 = BeautifulSoup(r2.content, 'html.parser')
    for i, t in enumerate(soup2.find_all('table')):
        rows = t.find_all('tr')
        print(f'\nTable {i:02d} — {len(rows)} rows')
        for row in rows[:6]:
            print('  ', row.get_text(separator=' | ', strip=True)[:100])

   prid=62890: 2026
   prid=62889: 2026
   prid=62888: 2026
   prid=62887: 2026
   prid=62886: 2026
   prid=62885: 2026
   prid=62884: 2026
   prid=62883: 2026
   prid=62882: 2026
   prid=62881: 2026
   prid=62880: 2026
   prid=62879: 2026
   prid=62878: 2026
   prid=62877: 2026
   prid=62876: 2026
   prid=62875: 2026
   prid=62874: 2026
   prid=62873: 2026
   prid=62872: 2026
   prid=62871: 2026
   prid=62869: 2026
   prid=62868: 2026
   prid=62867: 2026
   prid=62866: 2026
   prid=62865: 2026
   prid=62864: 2026
   prid=62863: 2026
   prid=62862: 2026
   prid=62861: 2026
   prid=62860: 2026
   prid=62859: 2026
   prid=62858: 2026
   prid=62857: 2026
   prid=62856: 2026
   prid=62855: 2026
   prid=62854: 2026
   prid=62853: 2026
   prid=62852: 2026
   prid=62851: 2026
   prid=62850: 2026
   prid=62848: 2026
   prid=62847: 2026
   prid=62846: 2026
   prid=62845: 2026
   prid=62844: 2026
   prid=62843: 2026
   prid=62842: 2026
   prid=62841: 2026

Found 0 forex reserve weekly releases i

In [34]:
# ── Cell 4 PROBE v4: Find correct title element, then re-scan ─────────────
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {'User-Agent': 'Mozilla/5.0'}

# Step 1: Inspect structure of one PR page
r = requests.get(
    'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid=62880',
    headers=HEADERS, timeout=15
)
soup = BeautifulSoup(r.content, 'html.parser')

print('=== ALL HEADINGS ===')
for tag in ['h1','h2','h3','h4','h5']:
    for el in soup.find_all(tag):
        txt = el.get_text(strip=True)
        if txt:
            print(f'  <{tag}>: {txt[:120]}')

print('\n=== PAGE <title> ===')
t = soup.find('title')
print(t.get_text(strip=True) if t else 'none')

print('\n=== LINES CONTAINING "reserve" or "week" ===')
for line in soup.get_text('\n').split('\n'):
    l = line.strip()
    if l and any(kw in l.lower() for kw in ['reserve', 'week ended', 'foreign exchange']):
        print(f'  {l[:120]}')

# Step 2: Re-scan using full page text search (not just heading tags)
print('\n\n=== RE-SCAN prid 62840-62890 using page text ===')
found_fx = []
for prid in range(62890, 62840, -1):
    url = f'https://www.rbi.org.in/Scripts/BS_PressReleaseDisplay.aspx?prid={prid}'
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            continue
        text = BeautifulSoup(r.content, 'html.parser').get_text(' ')
        # Find first non-empty meaningful line as title proxy
        lines = [l.strip() for l in text.split('\n') if len(l.strip()) > 20]
        title_proxy = lines[0][:100] if lines else 'empty'
        
        if any(kw in text.lower() for kw in ['week ended', 'foreign exchange reserves']):
            found_fx.append(prid)
            print(f'✅ prid={prid}: {title_proxy}')
        else:
            print(f'   prid={prid}: {title_proxy}')
        time.sleep(0.2)
    except Exception as e:
        print(f'   prid={prid}: ERROR {e}')

print(f'\nFound {len(found_fx)} forex reserve releases: {found_fx}')

=== ALL HEADINGS ===
  <h1>: Press Releases
  <h2>: 2026
  <h2>: 2025
  <h2>: 2024
  <h2>: 2023
  <h2>: 2022
  <h2>: 2021
  <h2>: 2020
  <h2>: 2019
  <h2>: 2018
  <h2>: 2017
  <h2>: Archives
  <h2>: 2016
  <h2>: 2015
  <h2>: 2014
  <h2>: 2013
  <h2>: 2012
  <h2>: 2011
  <h2>: 2010
  <h2>: 2009
  <h2>: 2008
  <h2>: 2007
  <h2>: 2006
  <h2>: 2005
  <h2>: 2004
  <h2>: 2003
  <h2>: 2002
  <h2>: 2001
  <h2>: 2000
  <h2>: 1999
  <h2>: 1998
  <h2>: 1997
  <h2>: 1996
  <h2>: 1995
  <h2>: 1994
  <h2>: 1993
  <h2>: 1992
  <h2>: 1991
  <h2>: 1990
  <h2>: MoreLinks
  <h2>: FollowRBI

=== PAGE <title> ===
Press Releases - Reserve Bank of India

=== LINES CONTAINING "reserve" or "week" ===
  Press Releases - Reserve Bank of India
  Reserve Bank Staff College
  The Central Government has re-appointed Shri Swaminathan Janakiraman as Deputy Governor, Reserve Bank of India for a per
  © Reserve Bank of India. All Rights Reserved.
  Website owned and managed by Reserve Bank of India. Contact us on helpdo

In [36]:
# ── Cell 4: RBI Forex Reserves (Weekly) ──────────────────────────────────
# Source : RBI Bulletin Table No. 33 — Foreign Exchange Reserves (Weekly)
# File   : data/rbi_forex_reserves_weekly.xlsx  (downloaded from RBI DBIE)
# Series : Total Reserves (US$ Millions) → converted to US$ Billions
# Frequency: Weekly (week-ended date). Forward-filled to daily in Cell 5.
#
# Coverage: April 2001 → May 2026 — full analysis window.

import pandas as pd
import numpy as np
from pathlib import Path

FX_RESERVES_PATH = DATA_DIR / 'rbi_forex_reserves_weekly.xlsx'

# ── Parse Excel ───────────────────────────────────────────────────────────
df_raw = pd.read_excel(FX_RESERVES_PATH, sheet_name=0, header=None)

# Rows where column 1 is a parseable date = actual data rows
# (skips year-band rows like "2025-26", section headers, notes)
col1 = df_raw.iloc[:, 1]
date_mask = pd.to_datetime(col1, errors='coerce').notna()
date_rows = df_raw[date_mask].copy()

dates      = pd.to_datetime(date_rows.iloc[:, 1], errors='coerce')
usd_mn     = pd.to_numeric(date_rows.iloc[:, 3], errors='coerce')   # col 3 = Total Reserves US$M

df_fx = pd.DataFrame({
    'date'                 : dates.values,
    'forex_reserves_usd_bn': (usd_mn / 1000).values          # US$M → US$B
}).dropna(subset=['date', 'forex_reserves_usd_bn'])

df_fx = df_fx.sort_values('date').reset_index(drop=True)
df_fx['date'] = pd.to_datetime(df_fx['date'])
df_fx = df_fx.set_index('date')
df_fx.index.name = 'date'

# ── Filter to analysis window ─────────────────────────────────────────────
df_fx = df_fx[df_fx.index >= START_DATE]

# ── Report ────────────────────────────────────────────────────────────────
print('RBI Forex Reserves loaded:')
print(f'  Observations : {len(df_fx)} weekly entries')
print(f'  Date range   : {df_fx.index.min().date()} → {df_fx.index.max().date()}')
print(f'  NaN count    : {df_fx["forex_reserves_usd_bn"].isna().sum()}')
print(f'\nSample (around policy date):')
mask = (df_fx.index >= '2026-04-01') & (df_fx.index <= '2026-06-01')
print(df_fx[mask].round(2).to_string())


RBI Forex Reserves loaded:
  Observations : 230 weekly entries
  Date range   : 2022-01-07 → 2026-05-29
  NaN count    : 0

Sample (around policy date):
            forex_reserves_usd_bn
date                             
2026-04-03                 697.12
2026-04-10                 700.95
2026-04-17                 703.31
2026-04-24                 698.49
2026-05-01                 690.69
2026-05-08                 696.99
2026-05-15                 688.89
2026-05-22                 681.38
2026-05-29                 682.32


In [38]:
# ── Cell 4b: RBI FX Intervention (Monthly) ───────────────────────────────
# Source : RBI Bulletin Table No. 04 — Sale/Purchase of U.S. Dollar by RBI
# File   : data/rbi_fx_intervention_monthly.xlsx
# Series : Net Purchase/Sale of Foreign Currency (US$ Millions)
#          Negative = RBI selling USD to defend rupee (reserve depletion)
#          Positive = RBI buying USD (reserve accumulation)
# Frequency : Monthly. Used for paper narrative (Section 2), not daily regression.
# Latest    : March 2026 (2-month publication lag — Apr/May 2026 unavailable).

FX_INTERVENTION_PATH = DATA_DIR / 'rbi_fx_intervention_monthly.xlsx'

df_int_raw = pd.read_excel(FX_INTERVENTION_PATH, sheet_name=0, header=None)

col1 = df_int_raw.iloc[:, 1]
date_mask = pd.to_datetime(col1, errors='coerce').notna()
date_rows = df_int_raw[date_mask].copy()

dates_int = pd.to_datetime(date_rows.iloc[:, 1], errors='coerce')
net_usd   = pd.to_numeric(date_rows.iloc[:, 3], errors='coerce')   # col 3 = net purchase/sale US$M

df_intervention = pd.DataFrame({
    'date'                   : dates_int.values,
    'rbi_net_usd_purchase_mn': net_usd.values
}).dropna()

df_intervention = df_intervention.sort_values('date').reset_index(drop=True)
df_intervention['date'] = pd.to_datetime(df_intervention['date'])

# ── Report ────────────────────────────────────────────────────────────────
print('RBI FX Intervention loaded:')
print(f'  Observations : {len(df_intervention)} monthly entries')
print(f'  Date range   : {df_intervention.date.min().date()} → {df_intervention.date.max().date()}')
print(f'  Latest month : {df_intervention.date.max().date()}  (Apr/May 2026 not yet published)')

print(f'\nPre-hike intervention story (Oct 2024 → Mar 2026):')
mask = (df_intervention.date >= '2024-10-01') & (df_intervention.date <= '2026-04-01')
print(df_intervention[mask][['date','rbi_net_usd_purchase_mn']].to_string(index=False))
print('\nNote: negative = RBI selling USD (defending rupee / burning reserves)')


RBI FX Intervention loaded:
  Observations : 337 monthly entries
  Date range   : 1995-06-30 → 2026-03-31
  Latest month : 2026-03-31  (Apr/May 2026 not yet published)

Pre-hike intervention story (Oct 2024 → Mar 2026):
      date  rbi_net_usd_purchase_mn
2024-10-31                  -9275.0
2024-11-30                 -20228.0
2024-12-31                 -15150.0
2025-01-31                 -11139.0
2025-02-28                  -1621.0
2025-03-31                  14355.0
2025-04-30                  -1660.0
2025-05-31                   1764.0
2025-06-30                  -3661.0
2025-07-31                  -2540.0
2025-08-31                  -7695.0
2025-09-30                  -7910.0
2025-10-31                 -11877.0
2025-11-30                  -9710.0
2025-12-31                 -10020.0
2026-01-31                   2526.0
2026-02-28                   7409.0
2026-03-31                  -9758.0

Note: negative = RBI selling USD (defending rupee / burning reserves)


In [37]:
# ── Cell 5: Alignment ────────────────────────────────────────────────────
# Purpose: Merge all fetched series onto one trading calendar.
#
# Three operations:
#   1. IBJA offset correction  — PDF dated D has rates for D-1 trading day
#                                → shift IBJA index back 1 business day
#   2. Merge                   — left join on Yahoo Finance date index (master)
#   3. Forex reserves ffill    — weekly RBI data forward-filled to daily

import pandas as pd
from pandas.tseries.offsets import BDay

# ── Step 1: IBJA 1-business-day offset correction ────────────────────────
# The PDF named "14-05-2026.pdf" (Wed) contains rates as of May 13 (Tue).
# The PDF named "19-05-2026.pdf" (Mon) contains rates as of May 16 (Fri).
# BDay(1) handles both cases correctly.

df_ibja_aligned = df_ibja.copy()
df_ibja_aligned.index = df_ibja_aligned.index - BDay(1)

# Drop any duplicates that arise if two PDFs map to the same business day
df_ibja_aligned = df_ibja_aligned[~df_ibja_aligned.index.duplicated(keep='last')]

# Offset sanity check around the policy date
print("IBJA offset check (PDF dates → corrected rate dates):")
print("  PDF May 13 → rate date May 12 (pre-hike, last normal trading day)")
print("  PDF May 14 → rate date May 13 (first post-hike rate)")
check_dates = pd.to_datetime(['2026-05-12', '2026-05-13', '2026-05-14', '2026-05-15'])
check = df_ibja_aligned[df_ibja_aligned.index.isin(check_dates)]
print(check[['Gold_INR_actual', 'ibja_source']].to_string())

# ── Step 2: Merge onto Yahoo Finance calendar ─────────────────────────────
# df_yf is the master — all dates from here survive the join
df = df_yf.copy()
df = df.join(
    df_ibja_aligned[['Gold_INR_actual', 'Silver_INR_actual', 'ibja_source']],
    how='left'
)

# ── Step 3: Forward-fill RBI forex reserves (weekly → daily) ─────────────
# Each Friday's reading is carried forward through the next week.
# Any dates after the last weekly observation (May 29) stay NaN — correct.
df_fx_daily = df_fx.reindex(df.index, method='ffill')
df = df.join(df_fx_daily[['forex_reserves_usd_bn']], how='left')

# ── Report ────────────────────────────────────────────────────────────────
print(f'\nMerged dataset:')
print(f'  Shape      : {df.shape}')
print(f'  Date range : {df.index.min().date()} → {df.index.max().date()}')

print(f'\nNaN counts after alignment:')
nan_df = df.isna().sum().rename('NaNs').to_frame()
nan_df['pct'] = (nan_df['NaNs'] / len(df) * 100).round(1).astype(str) + '%'
print(nan_df.to_string())

print(f'\nForex reserves around policy date:')
mask = (df.index >= '2026-04-25') & (df.index <= '2026-05-20')
print(df.loc[mask, ['rupees_per_dollar', 'Gold_INR_actual', 'forex_reserves_usd_bn']].to_string())


IBJA offset check (PDF dates → corrected rate dates):
  PDF May 13 → rate date May 12 (pre-hike, last normal trading day)
  PDF May 14 → rate date May 13 (first post-hike rate)
            Gold_INR_actual ibja_source
date                                   
2026-05-12         151954.0         pdf
2026-05-13         160411.0         pdf
2026-05-14         161349.0         pdf
2026-05-15         158159.0         pdf

Merged dataset:
  Shape      : (1157, 13)
  Date range : 2022-01-03 → 2026-06-10

NaN counts after alignment:
                       NaNs     pct
Oil_USD                  41    3.5%
Gold_USD                 42    3.6%
rupees_per_dollar         4    0.3%
Kalyan                   61    5.3%
Silver_USD               42    3.6%
Nifty50                  64    5.5%
US10Y_yield              44    3.8%
MCX_Gold               1157  100.0%
GOLDBEES                 62    5.4%
Gold_INR_actual         350   30.3%
Silver_INR_actual       350   30.3%
ibja_source             351   30.3%
fore

In [ ]:
# ── Cell 6: Construct premium and regression variables ────────────────────
#
# VERIFIED DUTY STRUCTURE (web-checked against TaxGuru, sunshinecargo.in,
# Business Standard, TaxCorp — all consistent):
#
#   Period              Dates                   BCD    AIDC   SWS   Total
#   ─────────────────── ──────────────────────  ─────  ─────  ────  ─────
#   High-duty           Jan 2022 – Jul 23 2024  10%    5%     0%    15%
#   Low-duty            Jul 24 2024 – May 12    5%     1%     0%    6%
#   Post-hike           May 13 2026+            10%    5%     0%    15%
#
# SWS = 0% throughout: gold bullion has a specific SWS exemption (confirmed
#   by Notification No. 16/2026 and Business Standard Jul 2024 budget coverage).
#   The headline "6% = 5% + 1%" leaves no room for SWS.
#
# Sources: taxguru.in (May 14 2026), sunshinecargo.in (May 13 2026),
#          thetaxcorp.in Notification 16/2026, business-standard.com Jul 2024.
#
# KEY DESIGN CHOICE:
#   parity_actual : time-varying duty (economically correct; used for EDA/plots)
#   parity_pre    : fixed at 6% (1.06) for ALL dates — the ITS regression
#                   baseline. It answers "what would gold cost if duty stayed
#                   at 6%?" Domestic_premium = how much market exceeds that.
#   parity_post   : fixed at 15% (1.15) — for comparing theoretical ceiling.
#
# IBJA benchmark is ex-GST (bullion-to-bullion), so IGST excluded throughout.

import numpy as np
import pandas as pd

# ── Duty multiplier constants ─────────────────────────────────────────────
DUTY_MULT_HIGH  = 1.15    # Jan 2022  – Jul 23 2024  (BCD 10% + AIDC 5%)
DUTY_MULT_LOW   = 1.06    # Jul 2024  – May 12 2026  (BCD 5%  + AIDC 1%)
DUTY_MULT_POST  = 1.15    # May 13 2026+             (BCD 10% + AIDC 5%)

TROY_OZ_TO_KG   = 1000 / 31.1035

print("=== Verified duty structure ===")
print(f"High-duty  (Jan 2022–Jul 2024) : BCD 10% + AIDC 5%  = {(DUTY_MULT_HIGH-1)*100:.1f}%  mult={DUTY_MULT_HIGH}")
print(f"Low-duty   (Jul 2024–May 2026) : BCD  5% + AIDC 1%  = {(DUTY_MULT_LOW-1)*100:.1f}%  mult={DUTY_MULT_LOW}")
print(f"Post-hike  (May 2026+)         : BCD 10% + AIDC 5%  = {(DUTY_MULT_POST-1)*100:.1f}%  mult={DUTY_MULT_POST}")
print(f"Duty shock : {DUTY_MULT_LOW} → {DUTY_MULT_POST}  (+{(DUTY_MULT_POST/DUTY_MULT_LOW-1)*100:.2f}% increase in landed cost)")

# ── Time-varying duty multiplier series ───────────────────────────────────
duty_mult_ts = pd.Series(DUTY_MULT_HIGH, index=df.index)
duty_mult_ts[df.index >= pd.Timestamp(DUTY_CUT_DATE)]  = DUTY_MULT_LOW
duty_mult_ts[df.index >= pd.Timestamp(POLICY_DATE)]    = DUTY_MULT_POST

gold_base = df['Gold_USD'] * TROY_OZ_TO_10G * df['rupees_per_dollar']

# ── Parity columns ────────────────────────────────────────────────────────
# parity_actual : uses the duty actually applicable on each date (EDA/plots)
df['parity_actual'] = gold_base * duty_mult_ts

# parity_pre  : fixed 6% baseline throughout — ITS regression denominator
df['parity_pre']    = gold_base * DUTY_MULT_LOW    # 1.06 everywhere

# parity_post : fixed 15% — theoretical ceiling for pass-through analysis
df['parity_post']   = gold_base * DUTY_MULT_POST   # 1.15 everywhere

# ── Gold premium (ITS outcome variable) ──────────────────────────────────
# Interpretation: how much does IBJA exceed the 6%-adjusted import price?
# Pre-hike low-duty window: should be near 0 (mkt ≈ 6% parity)
# Post-hike: should jump by ≈ parity_post − parity_pre ≈ ₹4,900/10g
df['domestic_premium'] = df['Gold_INR_actual'] - df['parity_pre']
df['premium_pct']      = (df['domestic_premium'] / df['parity_pre']) * 100

# ── Silver series (DESCRIPTIVE ONLY — not a valid placebo) ───────────────
# Silver duty also changed on May 13 2026 (same notifications, confirmed).
df['silver_parity_pre'] = (df['Silver_USD'] * TROY_OZ_TO_KG
                             * df['rupees_per_dollar'] * DUTY_MULT_LOW)
df['silver_premium'] = df['Silver_INR_actual'] - df['silver_parity_pre']

# ── Treatment variables ───────────────────────────────────────────────────
policy_ts = pd.Timestamp(POLICY_DATE)
df['post_hike']    = (df.index >= policy_ts).astype(int)

df['days_since_hike'] = np.nan
post_mask = df.index >= policy_ts
df.loc[post_mask, 'days_since_hike'] = np.arange(post_mask.sum())

# ── Change / return controls ──────────────────────────────────────────────
df['delta_Gold_USD'] = df['Gold_USD'].pct_change() * 100
df['delta_FX']       = df['rupees_per_dollar'].pct_change() * 100
df['delta_Oil']      = df['Oil_USD'].pct_change() * 100

# ── Sanity checks ─────────────────────────────────────────────────────────
cut_ts = pd.Timestamp(DUTY_CUT_DATE)
pre_cut  = df.loc[df.index <  cut_ts,  'domestic_premium'].dropna()
low_duty = df.loc[(df.index >= cut_ts) & (df.index < policy_ts), 'domestic_premium'].dropna()
post_hike= df.loc[df.index >= policy_ts, 'domestic_premium'].dropna()

print('\n=== Sub-period premium means (domestic_premium = IBJA − 6% parity) ===')
print(f'High-duty  (2022–Jul 2024): n={len(pre_cut):>4}  mean={pre_cut.mean():>8,.0f}  std={pre_cut.std():>6,.0f}')
print(f'Low-duty   (Jul24–May26) : n={len(low_duty):>4}  mean={low_duty.mean():>8,.0f}  std={low_duty.std():>6,.0f}')
print(f'Post-hike  (May26+)      : n={len(post_hike):>4}  mean={post_hike.mean():>8,.0f}  std={post_hike.std():>6,.0f}')

print('\n=== May 13, 2026 sanity check ===')
may13 = df.loc['2026-05-13']
ceiling = may13['parity_post'] - may13['parity_pre']
ibja_jump = 160411 - 151954   # confirmed IBJA: May 13 − May 12
print(f'  Gold_INR_actual  : {may13["Gold_INR_actual"]:,.0f}')
print(f'  parity_pre  (×1.06): {may13["parity_pre"]:,.0f}')
print(f'  parity_post (×1.15): {may13["parity_post"]:,.0f}')
print(f'  parity_actual (×1.15): {may13["parity_actual"]:,.0f}')
print(f'  domestic_premium    : {may13["domestic_premium"]:,.0f}  ({may13["premium_pct"]:.2f}%)')
print(f'  Theoretical ceiling (post−pre parity): {ceiling:,.0f} INR/10g')
print(f'  IBJA jump (May 12→13): +{ibja_jump:,}')
print(f'  Day-1 pass-through vs ceiling: {ibja_jump/ceiling:.1%}')

print(f'\n=== Column NaN check ===')
for col in ['parity_actual','parity_pre','parity_post','domestic_premium',
            'premium_pct','post_hike','days_since_hike']:
    n = df[col].isna().sum()
    print(f'  {col:<25} {n:>4} NaN ({n/len(df)*100:.1f}%)')


In [42]:
# ── Cell 7: Quality checks + save clean CSV ──────────────────────────────
# Step 1: Diagnose and fix extreme outliers in domestic_premium
# Step 2: Broad dataset health checks
# Step 3: Drop stale columns, save to data/gold_policy_clean.csv

import numpy as np
import pandas as pd

OUTPUT_PATH = DATA_DIR / 'gold_policy_clean.csv'

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1 — Outlier diagnostic & fix
# ═══════════════════════════════════════════════════════════════════════════
print('=== STEP 1: Outlier diagnostic & fix ===')

# ── Fix A: Zero Gold_INR_actual (PDF parse failure on 2022-10-26) ─────────
zero_ibja = df[df['Gold_INR_actual'] == 0.0].index
if len(zero_ibja) > 0:
    print(f'Zero Gold_INR_actual on {len(zero_ibja)} date(s): {[str(d.date()) for d in zero_ibja]}')
    for col in ['Gold_INR_actual', 'Silver_INR_actual', 'ibja_source',
                'domestic_premium', 'premium_pct', 'silver_premium']:
        df.loc[zero_ibja, col] = np.nan
    print('  → Nulled Gold_INR_actual and all IBJA-derived columns.')
else:
    print('No zero Gold_INR_actual found.')

# ── Fix B: Gold_USD futures-roll spikes (>5% single-day move) ────────────
gold_ret = df['Gold_USD'].pct_change().abs()
gold_spikes = df[gold_ret > 0.05].index
if len(gold_spikes) > 0:
    print(f'\nGold_USD spike dates ({len(gold_spikes)}): {[str(d.date()) for d in gold_spikes]}')
    for col in ['Gold_USD', 'parity_pre', 'parity_post',
                'domestic_premium', 'premium_pct',
                'silver_parity_pre', 'silver_premium', 'delta_Gold_USD']:
        df.loc[gold_spikes, col] = np.nan
    print('  → Nulled Gold_USD and all Gold_USD-derived columns.')
else:
    print('No Gold_USD spike dates found.')

# ── Verify outlier fix ────────────────────────────────────────────────────
prem = df['domestic_premium'].dropna()
print(f'\nPost-fix domestic_premium range: [{prem.min():,.0f}, {prem.max():,.0f}]')
print(f'Non-NaN observations: {len(prem)}')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 2 — Broad health checks
# ═══════════════════════════════════════════════════════════════════════════
print('\n=== STEP 2: Dataset health checks ===')
print(f'Shape (pre-clean): {df.shape}  ({df.index.min().date()} → {df.index.max().date()})')
print(f'All dates unique : {df.index.is_unique}')

weekend_count = (df.index.dayofweek >= 5).sum()
if weekend_count > 0:
    weekend_dates = df.index[df.index.dayofweek >= 5]
    print(f'Weekend dates ({weekend_count}): {[str(d.date()) for d in weekend_dates]}')
else:
    print('No weekend dates.')

print(f'\n--- NaN summary (final, before save) ---')
null_counts = df.isnull().sum().sort_values(ascending=False)
null_counts = null_counts[null_counts > 0]
for col, n in null_counts.items():
    print(f'  {col:<30} {n:>4}  ({n/len(df)*100:.1f}%)')

print(f'\n--- Value range checks ---')
checks = {
    'Gold_USD (USD/oz)':         (df['Gold_USD'],         500,   10000),
    'rupees_per_dollar':         (df['rupees_per_dollar'],  60,     110),
    'Gold_INR_actual (INR/10g)': (df['Gold_INR_actual'],  5000, 500000),
    'parity_pre':                (df['parity_pre'],        5000, 500000),
    'domestic_premium':         (df['domestic_premium'], -25000,  25000),
    'premium_pct (%)':           (df['premium_pct'],       -20,     20),
}
for label, (series, lo, hi) in checks.items():
    valid = series.dropna()
    out_of_range = ((valid < lo) | (valid > hi)).sum()
    flag = '  ⚠' if out_of_range > 0 else '  ✓'
    print(f'{flag}  {label:<35}  min={valid.min():>10,.1f}  max={valid.max():>10,.1f}  '
          f'out-of-range={out_of_range}')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 3 — Drop stale columns + save
# ═══════════════════════════════════════════════════════════════════════════
print('\n=== STEP 3: Drop stale columns & save ===')

# Stale columns created by the first (incorrect) version of Cell 6.
# The correct columns are parity_pre / parity_post / silver_parity_pre.
stale_cols = ['parity_6pct', 'parity_15pct', 'silver_parity_6pct']
cols_to_drop = [c for c in stale_cols if c in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f'Dropped stale columns: {cols_to_drop}')
else:
    print('No stale columns found.')

print(f'Final shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')

df.to_csv(OUTPUT_PATH)
print(f'\nSaved → {OUTPUT_PATH}')
print(f'File size: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB')

# ── Final policy window preview ───────────────────────────────────────────
print('\n--- Policy window (May 9–16, 2026) ---')
window = df.loc['2026-05-09':'2026-05-16',
                ['Gold_INR_actual', 'Gold_USD', 'rupees_per_dollar',
                 'parity_pre', 'domestic_premium', 'premium_pct', 'post_hike']]
print(window.to_string())

print('\n--- Pre/post summary ---')
policy_date = pd.Timestamp(POLICY_DATE)
pre  = df.loc[df.index < policy_date,  'domestic_premium'].dropna()
post = df.loc[df.index >= policy_date, 'domestic_premium'].dropna()
print(f'Pre-hike  n={len(pre):>4}  mean={pre.mean():>8,.0f}  std={pre.std():>7,.0f}  ' 
      f'min={pre.min():>9,.0f}  max={pre.max():>8,.0f}')
print(f'Post-hike n={len(post):>4}  mean={post.mean():>8,.0f}  std={post.std():>7,.0f}  '
      f'min={post.min():>9,.0f}  max={post.max():>8,.0f}')
print(f'Raw mean shift: +{post.mean() - pre.mean():,.0f} INR/10g')


=== STEP 1: Outlier diagnostic & fix ===
Zero Gold_INR_actual on 1 date(s): ['2022-10-26']
  → Nulled Gold_INR_actual and all IBJA-derived columns.
No Gold_USD spike dates found.

Post-fix domestic_premium range: [-14,293, 6,325]
Non-NaN observations: 769

=== STEP 2: Dataset health checks ===
Shape (pre-clean): (1157, 27)  (2022-01-03 → 2026-06-10)
All dates unique : True
Weekend dates (1): ['2025-02-01']

--- NaN summary (final, before save) ---
  MCX_Gold                       1157  (100.0%)
  days_since_hike                1136  (98.2%)
  premium_pct                     388  (33.5%)
  domestic_premium                388  (33.5%)
  silver_premium                  385  (33.3%)
  ibja_source                     351  (30.3%)
  Silver_INR_actual               351  (30.3%)
  Gold_INR_actual                 351  (30.3%)
  delta_Gold_USD                   89  (7.7%)
  delta_Oil                        83  (7.2%)
  Nifty50                          64  (5.5%)
  GOLDBEES                       